# 🔍 Praktikum 03 – Transformer-Architektur: Attention & Mini-Transformer
**Applied Generative AI – NLP | Sommersemester 2026**

> ⏱️ **Gesamtdauer: ~90 Minuten**  
> 🎯 **Lernziele:** Attention-Weights visualisieren · Welche Köpfe lernen was? · Single-Head Attention von Scratch · Causal Masking · Positional Encoding

---
### Pakete (einmalig)
```bash
pip install transformers bertviz torch matplotlib seaborn
```


In [ ]:
import importlib
import subprocess
import sys
from importlib.util import find_spec

IN_COLAB = "google.colab" in sys.modules
AUTO_INSTALL_MISSING = True

REQUIRED = {
    "transformers": "4.44.2",
    "bertviz": "1.4.0",
    "torch": "2.5.1",
    "matplotlib": "3.8.4",
    "seaborn": "0.13.2",
    "numpy": "1.26.4",
}

missing = [name for name in REQUIRED if find_spec(name) is None]
if missing:
    print("Fehlende Pakete:", ", ".join(missing))
    if AUTO_INSTALL_MISSING:
        specs = [f"{name}=={REQUIRED[name]}" for name in missing]
        print("Installiere:", ", ".join(specs))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *specs])
        print("Installation abgeschlossen. Bei Importfehlern Runtime neu starten.")
    else:
        print("Setze AUTO_INSTALL_MISSING=True oder installiere die Pakete manuell.")

for p in REQUIRED:
    status = "✅" if find_spec(p) else "❌"
    print(f"  {status}  {p}")

print("Runtime:", "Google Colab" if IN_COLAB else "Lokal/Jupyter")

## Teil 1 – Attention-Heatmaps mit BertViz ⏱️ ~25 min

BertViz ist eine spezialisierte Bibliothek zur Visualisierung von Attention-Gewichten  
in vortrainierten Transformer-Modellen. Wir verwenden `bert-base-uncased` als Beispiel.

### 1.1 – Modell und Tokenizer laden

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn as nn

MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModel.from_pretrained(MODEL_NAME, output_attentions=True)
model.eval()

print(f"Modell: {MODEL_NAME}")
print(f"  Encoder-Layer  : {model.config.num_hidden_layers}")
print(f"  Attention-Köpfe: {model.config.num_attention_heads}")
print(f"  Hidden Size    : {model.config.hidden_size}")
print(f"  Parameter      : {sum(p.numel() for p in model.parameters()):,}")

### 1.2 – BertViz: Interaktive Head-Visualisierung

In [ ]:
from bertviz import head_view, model_view

sentence = "The cat sat on the mat because it was comfortable."

inputs  = tokenizer(sentence, return_tensors="pt")
outputs = model(**inputs)

attention = outputs.attentions   # Tuple: (layer, batch, heads, seq, seq)
tokens    = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print(f"Satz: {sentence}")
print(f"Tokens ({len(tokens)}): {tokens}")
print(f"Attention-Shape: {len(attention)} Layer × {attention[0].shape}")


In [ ]:
# Head View: Welche Wörter beachtet welches Wort in welchem Kopf?
# ─────────────────────────────────────────────────────────────────
# Wenn in Jupyter Lab nicht interaktiv angezeigt:
# Nutze head_view(attention, tokens, layer=5) um nur einen Layer zu sehen

head_view(attention, tokens)


### 1.3 – Eigene Heatmap mit matplotlib

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def plot_attention_heatmap(attention, tokens, layer=5, head=0, title=None):
    attn = attention[layer][0][head].detach().numpy()

    fig, ax = plt.subplots(figsize=(len(tokens)*0.7 + 1, len(tokens)*0.7 + 1))

    mask = np.zeros_like(attn, dtype=bool)  # kein Mask bei BERT (bidirektional)

    sns.heatmap(attn, xticklabels=tokens, yticklabels=tokens,
                cmap="rocket_r", ax=ax, square=True,
                cbar_kws={"shrink": 0.5, "label": "Attention Weight"})

    ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(tokens, rotation=0, fontsize=9)
    ttl = title or f"Layer {layer+1} · Head {head+1}"
    ax.set_title(ttl, fontsize=13, fontweight="bold", pad=12)

    plt.tight_layout()
    plt.savefig(f"attention_L{layer+1}_H{head+1}.png", dpi=140, bbox_inches="tight")
    plt.show()
    return attn

# Layer 5, Kopf 1
attn = plot_attention_heatmap(attention, tokens, layer=5, head=0)


### 1.4 – Analyse: Was lernen verschiedene Köpfe?

> ❓ **Beobachtungsaufgabe:** Schau dir mehrere Layer/Köpfe an.  
> Kannst du folgende Muster identifizieren?  
> - **Syntaktische Abhängigkeiten** (Subjekt → Verb)  
> - **Lokale Attention** (jedes Token schaut auf Nachbarn)  
> - **Globale Attention** auf [CLS] oder [SEP]-Token

In [ ]:
# Vergleiche Layer 0 (früh) vs Layer 11 (spät) für mehrere Köpfe
sentence2 = "The bank can guarantee deposits will eventually cover future tuition costs."
# "bank" hat Doppelbedeutung → spannend für Attention!

inputs2  = tokenizer(sentence2, return_tensors="pt")
outputs2 = model(**inputs2)
attn2    = outputs2.attentions
tokens2  = tokenizer.convert_ids_to_tokens(inputs2["input_ids"][0])

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
fig.suptitle(f'"{sentence2}"\n', fontsize=11, fontweight="bold")

for col, head in enumerate(range(4)):
    for row, layer in enumerate([0, 11]):
        ax = axes[row][col]
        data = attn2[layer][0][head].detach().numpy()
        sns.heatmap(data, ax=ax, cmap="Blues", xticklabels=tokens2, yticklabels=tokens2,
                    cbar=False, square=True)
        ax.set_title(f"L{layer+1} H{head+1}", fontsize=9)
        ax.set_xticklabels(tokens2, rotation=90, fontsize=7)
        ax.set_yticklabels(tokens2, rotation=0, fontsize=7)
        ax.set_xlabel("")

plt.tight_layout()
plt.savefig("attention_multihead.png", dpi=130, bbox_inches="tight")
plt.show()
print("Hinweis: Schau wo 'bank' hinschaut – in Layer 0 vs. Layer 11!")


## Teil 2 – Single-Head Attention von Scratch ⏱️ ~30 min

### 2.1 – Die Formel verstehen

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

- **Q** (Query): "Was suche ich?"  
- **K** (Key): "Was biete ich an?"  
- **V** (Value): "Was gebe ich weiter?"

Jedes Token berechnet seine eigene Query, Key und Value durch lineare Projektionen.

In [ ]:
import torch.nn.functional as F
import math

class SingleHeadAttention(nn.Module):
    def __init__(self, d_model, causal=False):
        super().__init__()
        self.d_model = d_model
        self.causal  = causal

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x, return_weights=False):
        # x: (B, T, d_model)
        B, T, D = x.shape
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(D)   # (B, T, T)

        if self.causal:
            mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
            scores = scores.masked_fill(mask, float('-inf'))

        weights = F.softmax(scores, dim=-1)                 # (B, T, T)
        out     = weights @ V                               # (B, T, d_model)
        out     = self.W_o(out)

        if return_weights:
            return out, weights
        return out

# Sanity-Check
d_model = 64
seq_len = 10
batch   = 2
x       = torch.randn(batch, seq_len, d_model)

attn_causal     = SingleHeadAttention(d_model, causal=True)
attn_noncausal  = SingleHeadAttention(d_model, causal=False)

out_c, w_c  = attn_causal(x, return_weights=True)
out_nc, w_nc = attn_noncausal(x, return_weights=True)

print(f"Input shape       : {x.shape}")
print(f"Output shape      : {out_c.shape}")
print(f"Attention weights : {w_c.shape}")
print()
print("Causal – oberes Dreieck (sollte 0 sein):")
print(w_c[0].detach().numpy().round(3)[:5, :5])
print()
print("Non-Causal – i. A. nicht symmetrisch:")
print(w_nc[0].detach().numpy().round(3)[:5, :5])


### 2.2 – Causal Masking visualisieren

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

tokens_demo = [f"t{i}" for i in range(8)]
dummy_x = torch.randn(1, 8, 32)

attn_viz_c  = SingleHeadAttention(32, causal=True)
attn_viz_nc = SingleHeadAttention(32, causal=False)

with torch.no_grad():
    _, w_viz_c  = attn_viz_c(dummy_x, return_weights=True)
    _, w_viz_nc = attn_viz_nc(dummy_x, return_weights=True)

for ax, w, title in zip(axes,
                        [w_viz_nc[0].numpy(), w_viz_c[0].numpy()],
                        ["Bidirektionale Attention (BERT-Style)",
                         "Causale / Autoregressive Attention (GPT-Style)"]):
    sns.heatmap(w, ax=ax, cmap="Blues", square=True,
                xticklabels=tokens_demo, yticklabels=tokens_demo,
                annot=True, fmt=".2f", cbar=False, annot_kws={"size": 9})
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Key (von dem kommt Information)")
    ax.set_ylabel("Query (dieses Token fragt)")

plt.suptitle("Causal vs. Bidirektionale Attention", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("causal_vs_bidirectional.png", dpi=140, bbox_inches="tight")
plt.show()


## Teil 3 – Positional Encoding & Transformer-Block ⏱️ ~25 min

### 3.1 – Sinusoidales Positional Encoding

Self-Attention ohne Positionsinformation ist **permutationsäquivariant**: Bei vertauschter Reihenfolge vertauschen sich auch die Ausgaben.  
Positional Encodings fügen explizite Positions-Information hinzu.

$$PE_{(pos, 2i)}   = \sin\left(\frac{pos}{10000^{2i/d}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sinusoidal_pe(seq_len: int, d_model: int) -> torch.Tensor:
    """Sinusoidales Positional Encoding.
    
    Args:
        seq_len: Sequenzlänge
        d_model: Embedding-Dimension (sollte gerade sein für symmetrisches Design)
    """
    pe = torch.zeros(seq_len, d_model)
    position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    # Für ungerade d_model: Cos-Dimension kürzen
    pe[:, 1::2] = torch.cos(position * div_term[:d_model // 2])
    return pe

seq_len, d_model = 50, 128
pe = sinusoidal_pe(seq_len, d_model).numpy()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Heatmap: PE für alle Positionen und Dimensionen
im = axes[0].imshow(pe, aspect="auto", cmap="RdBu", interpolation="nearest")
axes[0].set_xlabel("Embedding-Dimension", fontsize=11)
axes[0].set_ylabel("Position im Satz", fontsize=11)
axes[0].set_title("Sinusoidales PE – Heatmap", fontsize=12, fontweight="bold")
plt.colorbar(im, ax=axes[0])

# Erste 4 Dimensionen über Positionen
for dim in range(0, 8, 2):
    axes[1].plot(pe[:, dim], label=f"dim {dim}")
axes[1].set_xlabel("Position", fontsize=11)
axes[1].set_ylabel("Encoding-Wert", fontsize=11)
axes[1].set_title("PE für die ersten Dimensionen", fontsize=12, fontweight="bold")
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("positional_encoding.png", dpi=140, bbox_inches="tight")
plt.show()


### 3.2 – Vollständiger Transformer-Block

In [ ]:
class FeedForward(nn.Module):
    """Position-wise FFN: 2 Linear-Layer mit GELU-Aktivierung."""
    def __init__(self, d_model: int, d_ff: int | None = None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    """Ein einzelner Transformer-Block (Pre-Norm Variante).

    Struktur: LayerNorm → Attention → Residual → LayerNorm → FFN → Residual
    """
    def __init__(self, d_model: int, causal: bool = True, dropout: float = 0.1):
        super().__init__()
        self.norm1  = nn.LayerNorm(d_model)
        self.attn   = SingleHeadAttention(d_model, causal=causal)
        self.norm2  = nn.LayerNorm(d_model)
        self.ff     = FeedForward(d_model)
        self.drop   = nn.Dropout(dropout)

    def forward(self, x):
        # Pre-Norm Residual-Attention
        x = x + self.drop(self.attn(self.norm1(x)))
        # Pre-Norm Residual-FFN
        x = x + self.drop(self.ff(self.norm2(x)))
        return x


# Kleines Sprachmodell mit 2 Transformer-Blöcken
class MiniGPT(nn.Module):
    def __init__(self, vocab_size=256, d_model=64, n_layers=2, max_len=64):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.register_buffer("pe", sinusoidal_pe(max_len, d_model), persistent=False)
        self.blocks = nn.ModuleList([TransformerBlock(d_model, causal=True) for _ in range(n_layers)])
        self.norm   = nn.LayerNorm(d_model)
        self.head   = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx):
        B, T = idx.shape
        x = self.embed(idx) + self.pe[:T].to(idx.device)
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        return self.head(x)  # (B, T, vocab_size) Logits


minigpt = MiniGPT()
total = sum(p.numel() for p in minigpt.parameters())
print(f"MiniGPT – Parameter: {total:,}")
print(f"Architektur:")
print(minigpt)

# Forward-Pass testen
idx = torch.randint(0, 256, (2, 32))   # 2 Sequenzen, Länge 32
logits = minigpt(idx)
print(f"\nInput:  {idx.shape}")
print(f"Output: {logits.shape}")  # (2, 32, 256) → pro Token 256 Logits

### 3.3 – Character-Level Language Model trainieren  
*(optional, ~10 min extra)*

Wir trainieren MiniGPT darauf, Zeichenfolgen zu modellieren.

In [ ]:
# Kurztext als Trainingsdaten (character-level)
corpus = """
Der Transformer ist eine revolutionäre Architektur für die Verarbeitung von Sequenzen.
Self-Attention ermöglicht es jedem Token, direkt mit jedem anderen Token zu interagieren.
Dies entschärft Probleme langer Abhängigkeiten in RNNs und ermöglicht Parallelisierung.
Moderne LLMs wie GPT-4, LLaMA und Claude basieren alle auf dieser Architektur.
""" * 20   # wiederholt für mehr Trainingsdaten

# Zeichenebene-Vokabular
chars  = sorted(set(corpus))
c2i    = {c: i for i, c in enumerate(chars)}
i2c    = {i: c for c, i in c2i.items()}
data   = torch.tensor([c2i[c] for c in corpus], dtype=torch.long)
vocab  = len(chars)

model_char = MiniGPT(vocab_size=vocab, d_model=64, n_layers=3, max_len=128)
opt   = torch.optim.Adam(model_char.parameters(), lr=3e-3)
crit  = nn.CrossEntropyLoss()

SEQ_LEN = 48
BATCH   = 32
STEPS   = 300

def get_batch():
    ix  = torch.randint(len(data) - SEQ_LEN - 1, (BATCH,))
    x   = torch.stack([data[i : i + SEQ_LEN]     for i in ix])
    y   = torch.stack([data[i + 1 : i + SEQ_LEN + 1] for i in ix])
    return x, y

losses = []
best_loss = float('inf')
patience = 20  # Early stopping: Breche ab, wenn keine Verbesserung mehr
patience_counter = 0

for step in range(STEPS):
    x, y = get_batch()
    logits = model_char(x)                       # (B, T, vocab)
    loss   = crit(logits.view(-1, vocab), y.view(-1))

    opt.zero_grad()
    loss.backward()
    opt.step()

    losses.append(loss.item())
    
    # Early stopping
    if loss.item() < best_loss:
        best_loss = loss.item()
        patience_counter = 0
    else:
        patience_counter += 1
    
    if (step + 1) % 50 == 0:
        print(f"Step {step+1:3d}/{STEPS} | Loss: {loss.item():.4f} | Patience: {patience_counter}/{patience}")
    
    if patience_counter >= patience:
        print(f"\nEarly stopping bei Step {step+1}. Keine Verbesserung seit {patience} Schritten.")
        break

plt.figure(figsize=(8,4))
plt.plot(losses, color="#2ecc71")
plt.title("MiniGPT Training Loss", fontweight="bold")
plt.xlabel("Step")
plt.ylabel("Cross-Entropy Loss")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("minigpt_training_loss.png", dpi=140)
plt.show()


In [ ]:
# Generierung – Greedy Decoding
def generate(model, start_text, max_new_tokens=80):
    model.eval()
    context = torch.tensor([c2i.get(c, 0) for c in start_text], dtype=torch.long).unsqueeze(0)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits  = model(context[:, -64:])          # Generation nutzt hier ein 64-Token-Fenster
            next_id = logits[0, -1].argmax().item()
            context = torch.cat([context,
                                  torch.tensor([[next_id]])], dim=1)
    return "".join([i2c[i.item()] for i in context[0]])

print("=== GENERATED TEXT ===")
print(generate(model_char, start_text="Der Transformer"))


## ✅ Zusammenfassung & Aufgaben

| Konzept | Umsetzung |
|---------|-----------|
| BertViz | Interaktive Attention-Heatmaps |
| Multi-Head Analyse | Layer 0 vs. Layer 11, verschiedene Köpfe |
| Causal vs. Bidirektional | Masking visualisiert |
| Attention from Scratch | Q, K, V, Skalierung, Softmax |
| Positional Encoding | Sinusoidal PE visualisiert |
| Transformer-Block | Pre-Norm + Residuals + FFN |
| MiniGPT | Character-Level Language Model |

### 🧩 Aufgaben zur Vertiefung

1. **Mehrdeutige Wörter:** Lade den Satz `"The bank by the river bank was crowded."` und schau, ob BERT die beiden "bank"-Tokens unterschiedlich kodiert (vergleiche Layer 0 vs. Layer 11).
2. **Multi-Head:** Erweitere `SingleHeadAttention` zu `MultiHeadAttention` mit `n_heads` Parametern.
3. **RoPE:** Recherchiere Rotary Position Embeddings (RoPE) – wie unterscheiden sie sich von sinusoidalen PEs?
